# Full-text criteria of the 20 reviews

Gets the section *Criteria for considering studies for this review* from the full text of each Cochrane review, so it can be compared with the LGAR criteria and the review abstract, and later used as review information.

Sources per review, in this order:
1. a Cochrane page saved from the browser in `data/raw/fulltext/cochrane/` (*Save Page As*, any file name). The review and its version are read from the page's `citation_doi` tag; a page with a different version than the DOI table is not used.
2. a file in `data/raw/fulltext/` named `<CD number>.html`, `.txt` or `.xml`
3. PMC full-text XML, which PMC returns only for open-access articles (2 of the 20)

Saving by hand is the working route: the Cochrane Library blocks scripted requests (Cloudflare).

Download (off by default): one plain `requests` call per review that has no saved file yet, 5 s apart, saved as `data/raw/fulltext/<CD number>.html`. Already saved reviews are skipped, so re-running is safe. It stops at the first response that is not the review page (e.g. a cookie or bot check) and does not try to get around it. Check the Cochrane Library / Wiley terms of use first.

Make sure each saved page is the version in the DOI table (e.g. `.pub3`), not a later update. The extraction warns if a saved page does not mention the expected DOI.

Code: [scripts/fulltext_criteria.py](../../scripts/fulltext_criteria.py). No paid API calls in this notebook. Files stay in `data/raw/` and `data/processed/`, which are not committed.

In [ ]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
import fulltext_criteria as F

DOWNLOAD = False         # request missing pages from the Cochrane Library (skips saved ones)
STOP_ON_FAILURE = True   # stop at the first response that is not the review page

## Reviews, DOIs and what is on disk

In [ ]:
abstracts, DOI = F.setup()
PAGES = F.saved_pages()
print(len(PAGES), "of", len(DOI), "reviews have a saved Cochrane page")
status = pd.DataFrame({
    "doi": DOI,
    "saved_page": {r: PAGES[r][0].name[:60] if r in PAGES else "" for r in DOI},
    "saved_version_ok": {r: (PAGES[r][1] == DOI[r]) if r in PAGES else None for r in DOI},
    "url": {r: f"https://www.cochranelibrary.com/cdsr/doi/{d}/full" for r, d in DOI.items()},
}).sort_index()
status

## Download

Each request prints status, redirects, final URL, server, page title and the start of the page text. The table below has the same per review. Responses that are not the review page are saved to `data/raw/fulltext/_failed/`.

In [ ]:
if DOWNLOAD:
    LOG = pd.DataFrame(F.download(DOI, stop_on_failure=STOP_ON_FAILURE)).set_index("review")
    display(LOG.drop(columns=["text_start"], errors="ignore"))

### Failed responses

In [ ]:
for path in sorted(F.FAILED.glob("*.html")):
    text = path.read_text(errors="replace")
    print(f"=== {path.name}: {len(text)} chars")
    print(text[:2000])
    print()

## Extract the criteria section

In [ ]:
FULLTEXT_CRITERIA, MISSING = F.extract(abstracts, DOI)

## Check

Length of the three versions of the review information per review, and one example in full.

In [ ]:
LGAR = json.loads((F.S.CRIT / "tar2019_intervention.json").read_text())
sel = {r: next((l for l in a["abstract"].split("\n") if l.startswith("SELECTION CRITERIA")), "") for r, a in abstracts.items()}
pd.DataFrame({
    "fulltext_chars": {r: len(v["criteria"]) for r, v in FULLTEXT_CRITERIA.items()},
    "lgar_chars": {r: len(c["inclusion_criteria"]) + len(c["exclusion_criteria"]) for r, c in LGAR.items()},
    "abstract_selection_chars": {r: len(s) for r, s in sel.items()},
    "source": {r: v["source"] for r, v in FULLTEXT_CRITERIA.items()},
}).sort_index()

In [ ]:
EXAMPLE = "CD012069" if "CD012069" in FULLTEXT_CRITERIA else next(iter(FULLTEXT_CRITERIA), None)
if EXAMPLE:
    print("=== full text", EXAMPLE, "\n", FULLTEXT_CRITERIA[EXAMPLE]["criteria"])
    print("\n=== LGAR\n", LGAR[EXAMPLE]["inclusion_criteria"], "\nEXCL:", LGAR[EXAMPLE]["exclusion_criteria"])
    print("\n=== abstract\n", sel[EXAMPLE])